In [ ]:
import json
import pandas as pd
from pathlib import Path
from IPython.display import display

In [ ]:
PROJECT_PATH = Path.cwd().parent
ARCHIVE_PATH = PROJECT_PATH / "archive"
DATA_PATH = PROJECT_PATH / "data"
MAPPINGS_PATH = PROJECT_PATH / "mappings"

print(PROJECT_PATH)
print(ARCHIVE_PATH)
print(DATA_PATH)
print(MAPPINGS_PATH)

In [ ]:
DATASET_FILE = "lunar_2024-10_2026-05.csv"

In [ ]:
# Initial inspection of data

def inspect_dataset(bank: str, filename: str) -> list[Path]:

    df = pd.read_csv(DATA_PATH / "raw" / bank / filename, sep=";")
    
    display(df.head(2))
    display(df.tail(2))
    print("Types: ")
    display(df.dtypes)
    print("\n")
    print("Nans: ")
    display(df.isna().sum().to_dict())
    print("\n")

    return df
    

DF = inspect_dataset("lunar", DATASET_FILE)

In [ ]:
# Inspect the columns that have many Nans. Exclude "Note" because all of its entries are Nan.
for c in ['Message', 'Card info', 'Sender', 'Receiver']:
    print(c)
    display(DF[DF[c].notna()])
    print("\n\n")

In [ ]:
# Inspect which columns have very few or just one unique entries. Those who do not provide something insightful wii be dropped.
for c in ['Text', 'Message', 'Transaction type', 'Card info', 'Currency', 'Sender', 'Receiver', 'Category']:
    print(c)
    display(DF[c].unique())
    print("\n\n")

In [ ]:
# Inspect this message cause it is a bit ambiguous in comparison to the rest
DF[DF["Message"] == "TX50455023700XT TICKETMASTER: 34131 TX50455023700XT, TX50455023700XT TICKETMASTER: 34131, TX50455023700XT"]

In [ ]:
# Before dropping columns, combine "Date" and "Time" into one column
DF["date"] = pd.to_datetime(DF["Date"] + " " + DF["Time"])
DF.head(2)

In [ ]:
# Drop columns. Later, once we finalize filling the "Category" column we will drop more
DF = DF.drop(columns=["Date", "Time", "Date of posting", "Time of posting", "Message", "Transaction type", "Card info"])
DF.head(2)

In [ ]:
# Extract entries from "Text" (this is the same as the "Description" column in other banks).
# Also, Lunar already provides the "Category" column, which is filled for some rows but Nan for others.
out_dir = ARCHIVE_PATH / "blank_mappings" / "lunar"
out_dir.mkdir(parents=True, exist_ok=True)

mapping = dict()

for _, c in DF.iterrows():
    text = c["Text"]
    cat = c["Category"]

    if text not in mapping or mapping[text] == "Not categorised":
        mapping[text] = cat

# # Sorting used for inspection. Will be omitted since .dump() can sort as well
# mapping = {k: mapping[k] for k in sorted(mapping)}


category_replacements = {
    "Not categorised": "misc",
    "Groceries": "Market | DK",
    "Transport": "Transport | DK",
    "Food and Drinks": "Going out",
}

mapping = {k: category_replacements.get(v, v) for k, v in mapping.items()}

with open(out_dir / "lunar_description_mapping.json", "w" , encoding="utf-8") as f:
    json.dump(mapping, f, sort_keys=True, ensure_ascii=False, indent=4)

In [ ]:
# Inspect generated JSON and manually categorise entries.

In [ ]:
# Upon inspection, it was observed that a "MobilePay" description (text) entry exist.
# There are 13 entries in the dataset that have this description.
# By manually checking the MobilePay app, we can find precise info about these transactions.
# Thus the steps followed will be:
# 1. Extract all the necessary info about those transaction in a JSON file
# 2. Inspect them and fill manually the "new_category" field with a more descriptive description
# 3. Load the completed new JSON file, locate the ambiguous "MobilePay" entries (based on date and index)
# 4. Replace ambiguous with new description

# Step 1: Extract MobilePay entries into a blank JSON for manual review
mobilepay_df = DF[DF["Text"] == "MobilePay"].sort_values("date")
print(f"Found {len(mobilepay_df)} MobilePay entries.")

mobilepay_misc_descriptions = []
for i, c in mobilepay_df.iterrows():
    mobilepay_misc_descriptions.append({
        "index": i,
        "text": c["Text"],
        "amount": c["Amount"],
        "date": str(c["date"]),
        "new_text": "",
        "category": ""
    })

out_file = ARCHIVE_PATH / "blank_mappings" / "lunar" / "lunar_mobilepay_mapping.json"

with open(out_file, "w", encoding="utf-8") as f:
    json.dump(mobilepay_misc_descriptions, f, ensure_ascii=False, indent=4)

print(f"Wrote {out_file.name}")

In [ ]:
# Steps 3 & 4: Load completed mappings and replace descriptions
# Run this AFTER manually filling in new_text and category fields

with open(MAPPINGS_PATH / "completed" / "lunar_mobilepay_mapping.json") as f:
    mobilepay_completed = json.load(f)

for entry in mobilepay_completed:
    condition = (
        (DF["Text"] == entry["text"]) &
        (DF["date"] == pd.to_datetime(entry["date"])) &
        (DF["Amount"] == entry["amount"])
    )
    DF.loc[condition, "Text"] = entry["new_text"]
    DF.loc[condition, "Category"] = entry["category"]


remaining = DF[DF["Text"] == "MobilePay"]
if remaining.empty:
    print(f"All {len(mobilepay_completed)} MobilePay entries replaced successfully.")
else:
    print(f"Warning: {len(remaining)} unresolved MobilePay rows remain.")

In [ ]:
# Clean and rename columns to match standard schema

DF = DF.drop(columns=["Sender", "Receiver", "Note"])
DF = DF.rename(columns={
    "Text": "description",
    "Amount": "amount",
    "Currency": "currency",
    "Category": "category",
    "Balance": "balance",
})
DF["source"] = DATASET_FILE
print(f"Columns: {DF.columns.tolist()}")
print(f"Rows: {len(DF)}")
DF.head()

In [ ]:
# Attach categories from completed mapping

with open(MAPPINGS_PATH / "completed" / "lunar.json", encoding="utf-8") as f:
    category_mapping = json.load(f)

DF["category"] = DF["description"].map(category_mapping).fillna(DF["category"])

# For any remaining NaN categories
DF["category"] = DF["category"].fillna("misc")

unmapped = DF[DF["category"] == "misc"]["description"].unique()

print(f"Unmapped descriptions: {len(unmapped)}")

for d in sorted(unmapped):
    print(f"  {d}")

In [ ]:
# Parse amount and balance (European number format)
for col in ["amount", "balance"]:
    DF[col] = (
        DF[col]
        .str.replace(".", "", regex=False)
        .str.replace(",", ".", regex=False)
        .astype(float)
    )
print(DF.dtypes)

In [ ]:
# Add EUR conversion column
DKK_TO_EUR = 7.46
DF["amount_euro"] = (DF["amount"] / DKK_TO_EUR).round(2)
DF.head()

In [ ]:
# Attach categories from completed mapping
with open(MAPPINGS_PATH / "completed" / "lunar.json", encoding="utf-8") as f:
    category_mapping = json.load(f)

DF["category"] = DF["description"].map(category_mapping).fillna(DF["category"])

# For any remaining NaN categories
DF["category"] = DF["category"].fillna("misc")

unmapped = DF[DF["category"] == "misc"]["description"].unique()

print(f"Unmapped descriptions: {len(unmapped)}")

for d in sorted(unmapped):
    print(f"  {d}")

In [ ]:
# Sanity check
print(f"Total rows: {len(DF)}", end='\n\n')
print(f'Min date: {DF["date"].min()}', end='\n\n')
print(f'Max date: {DF["date"].max()}', end='\n\n')
print(f'NaNs: {DF.isna().sum().to_dict()}', end='\n\n')
print(f'Categories: {DF["category"].nunique()} unique', end='\n\n')
print(f"\nCategory breakdown:")
print(DF["category"].value_counts().sort_index().to_string())